In [111]:
pip install fredapi python-dotenv pandas statsmodels


  Using cached scipy-1.17.1-cp314-cp314-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 5.4 MB/s  0:00:02eta 0:00:01
Using cached patsy-1.0.2-py2.py3-none-any.whl (233 kB)
Using cached scipy-1.17.1-cp314-cp314-macosx_14_0_arm64.whl (20.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [statsmodels] [statsmodels]

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [71]:
from fredapi import Fred
from dotenv import load_dotenv
import pandas as pd
import os
from functools import reduce

In [7]:
load_dotenv()
FRED_API_KEY = os.getenv("FRED_API_KEY")
fred_auth = fred = Fred(api_key=FRED_API_KEY)

# Point In Time Logic (Predicting on a monthly time unit)

* If your model is **designed to run on the 1st of the month** (e.g., Jan 1st), it has to use whatever the most recently published data is on that exact day.
* When we train our model using historical data, we are not just trying to find historical patterns. We are trying to simulate exactly what the model would have seen if it had been running live on that historical day.

### Example
* When we are training the model, its supposed to predict on a time period of one month (let's say the 31st of each month)
* When it predicts on the 31st of the month you are try to fetch all the data, but for january's unemployment rate, cpi, etc. all of these things aren't available until mid feb
* So actually, you are supposed to use last month's data to do the prediction (e.g. the december data is used for january prediction)

In [94]:
recession_data = pd.DataFrame(fred_auth.get_series("USREC")).reset_index().rename(columns={"index": "aligned_date", 0: "recession"})
recession_data = recession_data[recession_data['aligned_date'] >= pd.to_datetime('2001-01-01')]

# list of metrics to pull from FRED: consumer price index, payroll employment, unemployment rate, industrial production, 10 year minus 3 month treasury yield
# 'realtime_start' = The Publication Day / Knowledge Date
# 'date' = The Observation Month
# 'value' = The CPI value at that time

monthly_revised = ["CPIAUCSL", "PAYEMS", "UNRATE", "INDPRO"]
daily_market = ["T10Y3M"]

economic_data = {}

economic_data["recession"] = recession_data

for metric in monthly_revised:
    print(f"Pulling first-release data for {metric}...")
    df = fred_auth.get_series_all_releases(metric, realtime_start="2001-01-01")
    df = df[df['date'] >= pd.to_datetime('2001-01-01')]
    # The reason for first releases is because we are assessing the data as it was available at the time of release
    # because this often guides future economic decisions and market reactions.
    # we also only keep the 'realtime_start' and 'value' columns, as we our focusing on the publication day
    first_release_df = df.groupby("date").head(1)
    first_release_df["aligned_date"] = pd.to_datetime(first_release_df["realtime_start"]) + pd.offsets.MonthBegin(1)
    first_release_df.rename(columns={"realtime_start": "publication_date",'date': 'observation_month','value': f"{metric}"}, inplace=True)
    economic_data[metric] = first_release_df


for metric in daily_market:
    print(f"Pulling market data for {metric}...")
    economic_data[metric] = fred_auth.get_series(metric, observation_start="2001-01-01")
    # resample for last day of the month
    # convert to period and back to timestamp to get the last day of the month, 
    # reset index and rename columns
    economic_data[metric] = (
        pd.DataFrame(economic_data[metric])
        .resample("ME")
        .last()
        .reset_index()
        .rename(columns={"index": "date", 0: f"{metric}"})
    )
    economic_data[metric]["aligned_date"] = economic_data[metric]["date"] + pd.offsets.MonthBegin(1)

Pulling first-release data for CPIAUCSL...
Pulling first-release data for PAYEMS...
Pulling first-release data for UNRATE...
Pulling first-release data for INDPRO...
Pulling market data for T10Y3M...


- The model now assumes we are running on the [first of each month]
- We pull the most 'recent' available data for cpi/unemployment date, which is the data published on the month before - which was observing for the month before that
- We then also pull the last known yield curve data excluding the day we are running it
- Which is why we have aligned dates for all the data sources which is offset to the first day of the month after the data was published

In [103]:
all_dfs_cleaned = [df[['aligned_date', metric]] for metric, df in economic_data.items()]
df_final = reduce(lambda left, right: pd.merge(left, right, on='aligned_date', how='outer'), all_dfs_cleaned).sort_values('aligned_date')
df_final.dropna(inplace=True)
df_final.tail(20)

,aligned_date,recession,CPIAUCSL,PAYEMS,UNRATE,INDPRO,T10Y3M
287,2024-12-01,0.0,315.454,159005.0,4.1,102.2805,-0.40
288,2025-01-01,0.0,316.441,159288.0,4.2,101.9621,0.21
289,2025-02-01,0.0,317.685,159536.0,4.1,103.1942,0.27
290,2025-03-01,0.0,319.086,159069.0,4.0,103.511,-0.08
291,2025-04-01,0.0,319.775,159218.0,4.1,104.2062,-0.09
292,2025-05-01,0.0,319.615,159398.0,4.2,103.8892,-0.14
293,2025-06-01,0.0,320.321,159517.0,4.2,103.8799,0.05
294,2025-07-01,0.0,320.58,159561.0,4.2,103.5948,-0.17
295,2025-08-01,0.0,321.5,159724.0,4.1,104.0071,-0.04
296,2025-09-01,0.0,322.132,159539.0,4.2,103.9867,0.00


In [108]:
# target variable engineering - for month t, our target y_t is 1 if there is a recession in any of the subsequent 6 months
df_final['target'] = df_final['recession'].rolling(window=6, min_periods=1).max().shift(-5)

# For trending cumulative data (CPIAUCSL, PAYEMS, INDPRO): 
# Calculate the Month-over-Month (MoM) percentage change and the Year-over-Year (YoY) percentage change (which is a 12-month lookback).
for metric in ["CPIAUCSL", "PAYEMS", "INDPRO"]:
    df_final[f"{metric}_MoM"] = df_final[metric].pct_change()
    df_final[f"{metric}_YoY"] = df_final[metric].pct_change(periods=12)
    
# For rate-based data (UNRATE, T10Y3M): These are already percentages. 
# Instead of a percentage of a percentage, it is standard practice to just calculate the absolute 
# difference from the previous month (MoM Difference) and from 12 months ago (YoY Difference).
for metric in ["UNRATE", "T10Y3M"]:
    df_final[f"{metric}_MoM_diff"] = df_final[metric].diff()
    df_final[f"{metric}_YoY_diff"] = df_final[metric].diff(periods=12)

In [109]:
df_final.tail(20)

,aligned_date,recession,CPIAUCSL,PAYEMS,UNRATE,INDPRO,T10Y3M,target,CPIAUCSL_MoM,CPIAUCSL_YoY,PAYEMS_MoM,PAYEMS_YoY,INDPRO_MoM,INDPRO_YoY,UNRATE_MoM_diff,UNRATE_YoY_diff,T10Y3M_MoM_diff,T10Y3M_YoY_diff
287,2024-12-01,0.0,315.454,159005.0,4.1,102.2805,-0.40,0.0,0.002441,0.02547,-0.000629,0.013268,-0.00352,-0.004163,0.0,0.2,-0.04,0.68
288,2025-01-01,0.0,316.441,159288.0,4.2,101.9621,0.21,0.0,0.003129,0.027683,0.00178,0.014011,-0.003113,-0.006848,0.1,0.5,0.61,1.73
289,2025-02-01,0.0,317.685,159536.0,4.1,103.1942,0.27,0.0,0.003931,0.028606,0.001557,0.014654,0.012084,0.00693,-0.1,0.4,0.06,1.70
290,2025-03-01,0.0,319.086,159069.0,4.0,103.511,-0.08,0.0,0.00441,0.030357,-0.002927,0.008681,0.00307,0.009136,-0.1,0.3,-0.35,1.12
291,2025-04-01,0.0,319.775,159218.0,4.1,104.2062,-0.09,0.0,0.002159,0.028037,0.000937,0.008935,0.006716,0.018322,0.1,0.2,-0.01,1.17
292,2025-05-01,0.0,319.615,159398.0,4.2,103.8892,-0.14,0.0,-0.0005,0.023652,0.001131,0.008,-0.003042,0.011996,0.1,0.4,-0.05,0.63
293,2025-06-01,0.0,320.321,159517.0,4.2,103.8799,0.05,0.0,0.002209,0.022713,0.000747,0.007777,-0.00009,0.010876,0.0,0.3,0.19,1.00
294,2025-07-01,0.0,320.58,159561.0,4.2,103.5948,-0.17,0.0,0.000809,0.023482,0.000276,0.006421,-0.002745,0.002578,0.0,0.2,-0.22,0.95
295,2025-08-01,0.0,321.5,159724.0,4.1,104.0071,-0.04,0.0,0.00287,0.026996,0.001022,0.006846,0.00398,0.000125,-0.1,0.0,0.13,1.28
296,2025-09-01,0.0,322.132,159539.0,4.2,103.9867,0.00,0.0,0.001966,0.027423,-0.001158,0.005141,-0.000196,0.010672,0.1,-0.1,0.04,1.30


# Augmented Dickey-Fuller (ADF) 
It is a test for stationarity - a time series is stationary if its core statistical properties—specifically its mean and variance—do not change over time.

- Imagine a pendulum swinging back and forth around a fixed center point. That is stationary.
- Now imagine a dog wandering freely through a park, wandering further and further away. That is a "random walk," which is non-stationary.

Macroeconomic absolute numbers (like total CPI or the S&P 500 price) are non-stationary. They drift upward over decades. If you feed them into a machine learning model, the model essentially learns "higher number = closer to the year 2024," which is completely useless for predicting the future.

**ADF test is fundamentally checking for something called a "Unit Root." If a time series has a unit root, it is a random walk (non-stationary).**

Let's look at a basic Autoregressive, or AR(1), model, which states that today's value ($y_t$) depends on yesterday's value ($y_{t-1}$) plus some random noise ($\epsilon_t$):$$y_t = \rho y_{t-1} + \epsilon_t$$

If $\rho = 1$, the series has a "unit root." This means today's value is exactly yesterday's value plus random noise. It will drift infinitely.

If you subtract $y_{t-1}$ from both sides to look at the change ($\Delta y_t$):$$\Delta y_t = (\rho - 1)y_{t-1} + \epsilon_t$$Let's define $\gamma = \rho - 1$. The equation becomes:$$\Delta y_t = \gamma y_{t-1} + \epsilon_t$$

Test hypotheses:
- The Null Hypothesis ($H_0$): $\gamma = 0$ (which means $\rho = 1$). The series has a unit root and is non-stationary.
- The Alternative Hypothesis ($H_1$): $\gamma < 0$. The series does not have a unit root and is stationary.


In [113]:
# The Augmented Dickey-Fuller test is a formal statistical test where the null hypothesis is that the time series is non-stationary.
# If the p-value is < 0.05, you reject the null hypothesis and can confidently tell your evaluators: "My features are stationary."
from statsmodels.tsa.stattools import adfuller
for column in df_final.columns:
    if column not in ['aligned_date', 'recession', 'target',"CPIAUCSL", "PAYEMS", "INDPRO", "UNRATE", "T10Y3M"]:
        result = adfuller(df_final[column].dropna())
        print(f"ADF Statistic for {column}: {result[0]}")
        print(f"p-value for {column}: {result[1]}")
        print("------")

ADF Statistic for CPIAUCSL_MoM: -10.58751089885634
p-value for CPIAUCSL_MoM: 6.657814746912218e-19
------
ADF Statistic for CPIAUCSL_YoY: -2.8761122732786366
p-value for CPIAUCSL_YoY: 0.048197166376869535
------
ADF Statistic for PAYEMS_MoM: -14.578694399217088
p-value for PAYEMS_MoM: 4.5051182952814734e-27
------
ADF Statistic for PAYEMS_YoY: -3.2903675863654844
p-value for PAYEMS_YoY: 0.015314572574978758
------
ADF Statistic for INDPRO_MoM: -15.750245402319063
p-value for INDPRO_MoM: 1.2257927414459716e-28
------
ADF Statistic for INDPRO_YoY: -3.979379963462776
p-value for INDPRO_YoY: 0.0015224170542714383
------
ADF Statistic for UNRATE_MoM_diff: -10.29527033618784
p-value for UNRATE_MoM_diff: 3.484492161048111e-18
------
ADF Statistic for UNRATE_YoY_diff: -3.4026653140779666
p-value for UNRATE_YoY_diff: 0.010868346114985593
------
ADF Statistic for T10Y3M_MoM_diff: -9.777735910259763
p-value for T10Y3M_MoM_diff: 6.862902186324655e-17
------
ADF Statistic for T10Y3M_YoY_diff: -3.21